In [8]:
# scripts/02_scottish_event_profiler.py
import requests
import polars as pl
import os
from datetime import datetime

# ==============================================================================
# MILESTONE 2.2: SCOTTISH EVENT PROFILER
# Goal: Calculate actual duration and frequency of Scottish export constraint events
# ==============================================================================

API_URL = "https://api.neso.energy/api/3/action/datastore_search"
RESOURCE_ID = "38a18ec1-9e40-465d-93fb-301e80fd1352"  # Day Ahead Constraint Flows
TIMEOUT = (5, 10)

TARGET_BOUNDARIES = ["SCOTEX", "SSEN-S"]

def fetch_boundary_data(boundary: str) -> pl.DataFrame:
    """
    Fetch all data for a specific boundary.
    Uses pagination to handle large datasets.
    """
    print(f"\nFetching data for {boundary}...")
    
    all_records = []
    offset = 0
    limit = 10000  # Fetch in chunks
    
    while True:
        params = {
            "resource_id": RESOURCE_ID,
            "limit": limit,
            "offset": offset,
            "filters": f'{{"Constraint Group": "{boundary}"}}'
        }
        
        try:
            response = requests.get(API_URL, params=params, timeout=TIMEOUT)
            response.raise_for_status()
            data = response.json()
            
            if not data.get("success"):
                raise RuntimeError(f"API error: {data.get('error')}")
            
            records = data["result"]["records"]
            
            if not records:
                break
                
            all_records.extend(records)
            offset += limit
            
            total = data["result"].get("total", len(all_records))
            print(f"  Fetched {len(all_records)} of {total} rows", end='\r')
            
            if len(all_records) >= total:
                break
                
        except Exception as e:
            print(f"\n  ❌ Failed at offset {offset}: {e}")
            break
    
    print(f"\n  ✅ Fetched {len(all_records)} total rows for {boundary}")
    
    if all_records:
        df = pl.DataFrame(all_records)
        
        # Parse dates and convert to numeric with explicit DST handling
                # Parse dates and convert to numeric with explicit DST handling
        # Chain transformations inline — Polars evaluates .with_columns() in parallel,
        # so we cannot reference aliases created in the same block.
        df = df.with_columns([
            # Single chained expression: parse → localize → convert to UTC
            pl.col("Date (GMT/BST)")
              .str.to_datetime(strict=False)
              .dt.replace_time_zone(
                  "Europe/London",
                  ambiguous="earliest",
                  non_existent="null"
              )
              .dt.convert_time_zone("UTC")
              .alias("timestamp"),

            # Cast metrics
            pl.col("Limit (MW)").cast(pl.Float64, strict=False),
            pl.col("Flow (MW)").cast(pl.Float64, strict=False),
        ])

        # Filter out any null timestamps (spring-forward missing hours)
        df = df.filter(pl.col("timestamp").is_not_null())
        
        # Calculate constraint excess and volume
        df = df.with_columns([
            (pl.col("Flow (MW)") - pl.col("Limit (MW)")).alias("excess_mw"),
            ((pl.col("Flow (MW)") - pl.col("Limit (MW)")) * 0.5).alias("constraint_volume_mwh")
        ])
        
        return df
    else:
        return pl.DataFrame()

def identify_constraint_events(df: pl.DataFrame) -> pl.DataFrame:
    """
    Identify constraint events (where Flow > Limit).
    Groups consecutive half-hours into single events.
    
    NOTE: Per PROJECT.md §4.3, each .with_columns() block is evaluated in parallel.
    Column aliases created in one block can only be referenced in SUBSEQUENT blocks.
    """
    print("\nIdentifying constraint events...")
    
    # Filter to only constraint periods
    constraint_df = df.filter(pl.col("excess_mw") > 0)
    
    if constraint_df.is_empty():
        print("  ⚠️ No constraint events found")
        return pl.DataFrame()
    
    # Sort by timestamp
    constraint_df = constraint_df.sort("timestamp")
    
    # BLOCK 1: Calculate gap in minutes (chain .diff() → .dt.total_minutes() inline)
    # ❌ WRONG (violates §4.3):
    #     pl.col("timestamp").diff().alias("time_diff"),
    #     pl.col("time_diff").dt.total_minutes().alias("diff_minutes")
    # ✅ CORRECT: chain directly, no intermediate alias
    constraint_df = constraint_df.with_columns([
        pl.col("timestamp")
          .diff()
          .dt.total_minutes()
          .alias("diff_minutes")
    ])
    
    # BLOCK 2: Mark new events (references diff_minutes from BLOCK 1 — allowed)
    constraint_df = constraint_df.with_columns([
        (pl.col("diff_minutes") > 30).alias("new_event")
    ])
    
    # BLOCK 3: Handle first row (references new_event from BLOCK 2 — allowed)
    constraint_df = constraint_df.with_columns([
        pl.col("new_event").fill_null(True)
    ])
    
    # BLOCK 4: Create event IDs via cumulative sum
    constraint_df = constraint_df.with_columns([
        pl.col("new_event").cum_sum().alias("event_id")
    ])
    
    # BLOCK 5: Aggregate by event (all columns now exist from prior blocks)
    events = constraint_df.group_by("event_id").agg([
        pl.col("timestamp").min().alias("start_time"),
        pl.col("timestamp").max().alias("end_time"),
        pl.col("excess_mw").mean().alias("avg_excess_mw"),
        pl.col("excess_mw").max().alias("max_excess_mw"),
        pl.col("constraint_volume_mwh").sum().alias("total_volume_mwh"),
        pl.len().alias("duration_half_hours")  # Use pl.len() instead of deprecated pl.count()
    ])
    
    # Calculate duration in hours
    events = events.with_columns([
        (pl.col("duration_half_hours") * 0.5).alias("duration_hours")
    ])
    
    print(f"  ✅ Identified {len(events)} constraint events")
    
    return events

def calculate_statistics(events: pl.DataFrame, boundary: str) -> dict:
    """
    Calculate summary statistics for constraint events.
    """
    if events.is_empty():
        return {}
    
    stats = {
        "boundary": boundary,
        "total_events": len(events),
        "total_volume_mwh": events["total_volume_mwh"].sum(),
        "avg_duration_hours": events["duration_hours"].mean(),
        "median_duration_hours": events["duration_hours"].median(),
        "max_duration_hours": events["duration_hours"].max(),
        "min_duration_hours": events["duration_hours"].min(),
        "p10_duration_hours": events["duration_hours"].quantile(0.10),
        "p50_duration_hours": events["duration_hours"].quantile(0.50),
        "p90_duration_hours": events["duration_hours"].quantile(0.90),
        "events_under_4h": (events["duration_hours"] < 4).sum(),
        "pct_under_4h": (events["duration_hours"] < 4).sum() / len(events) * 100,
    }
    
    return stats

if __name__ == "__main__":
    os.makedirs("data/intermediate", exist_ok=True)
    os.makedirs("data/processed", exist_ok=True)
    
    print("=" * 80)
    print("MILESTONE 2.2: SCOTTISH EVENT PROFILER")
    print("=" * 80)
    
    all_events = []
    all_stats = []
    
    for boundary in TARGET_BOUNDARIES:
        print(f"\n{'─' * 80}")
        print(f"Processing: {boundary}")
        print(f"{'─' * 80}")
        
        # Fetch data
        df = fetch_boundary_data(boundary)
        
        if df.is_empty():
            print(f"  ⚠️ No data for {boundary}")
            continue
        
        # Save raw data
        raw_path = f"data/intermediate/02_{boundary}_raw_data.parquet"
        df.write_parquet(raw_path)
        print(f"  📁 Raw data saved to: {raw_path}")
        
        # Identify events
        events = identify_constraint_events(df)
        
        if not events.is_empty():
            # Add boundary column
            events = events.with_columns(pl.lit(boundary).alias("boundary"))
            all_events.append(events)
            
            # Calculate statistics
            stats = calculate_statistics(events, boundary)
            all_stats.append(stats)
            
            # Save events
            events_path = f"data/intermediate/02_{boundary}_constraint_events.parquet"
            events.write_parquet(events_path)
            print(f"  📁 Events saved to: {events_path}")
    
    # Combine all events
    if all_events:
        all_events_df = pl.concat(all_events)
        combined_events_path = "data/processed/02_scottish_constraint_events.parquet"
        all_events_df.write_parquet(combined_events_path)
        print(f"\n📁 Combined events saved to: {combined_events_path}")
    
    # Print summary statistics
    print("\n" + "=" * 80)
    print("SCOTTISH CONSTRAINT EVENT STATISTICS")
    print("=" * 80)
    
    for stats in all_stats:
        print(f"\n{stats['boundary']}:")
        print(f"  Total events: {stats['total_events']}")
        print(f"  Total constraint volume: {stats['total_volume_mwh']:,.0f} MWh")
        print(f"  Average duration: {stats['avg_duration_hours']:.2f} hours")
        print(f"  Median duration: {stats['median_duration_hours']:.2f} hours")
        print(f"  Duration distribution:")
        print(f"    P10: {stats['p10_duration_hours']:.2f} hours")
        print(f"    P50: {stats['p50_duration_hours']:.2f} hours")
        print(f"    P90: {stats['p90_duration_hours']:.2f} hours")
        print(f"  Events under 4 hours: {stats['events_under_4h']} ({stats['pct_under_4h']:.1f}%)")
    
    print("\n" + "=" * 80)
    print("✅ MILESTONE 2.2 COMPLETE")
    print("=" * 80)        

MILESTONE 2.2: SCOTTISH EVENT PROFILER

────────────────────────────────────────────────────────────────────────────────
Processing: SCOTEX
────────────────────────────────────────────────────────────────────────────────

Fetching data for SCOTEX...
  Fetched 61066 of 61066 rows
  ✅ Fetched 61066 total rows for SCOTEX
  📁 Raw data saved to: data/intermediate/02_SCOTEX_raw_data.parquet

Identifying constraint events...
  ✅ Identified 803 constraint events
  📁 Events saved to: data/intermediate/02_SCOTEX_constraint_events.parquet

────────────────────────────────────────────────────────────────────────────────
Processing: SSEN-S
────────────────────────────────────────────────────────────────────────────────

Fetching data for SSEN-S...
  Fetched 60484 of 60484 rows
  ✅ Fetched 60484 total rows for SSEN-S
  📁 Raw data saved to: data/intermediate/02_SSEN-S_raw_data.parquet

Identifying constraint events...
  ✅ Identified 1023 constraint events
  📁 Events saved to: data/intermediate/02_SSE